# Challenge 6 · Fight the Noise
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hastikacheddy/qiskit-fall-fest-participant/blob/main/challenges/challenge_06_noise/challenge_06_noise.ipynb)

**Tier 2 - Quantum Algorithms and Applications** · **Estimated time: 40-60 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Your ideal circuit works perfectly - on paper. Real devices are noisy. Quantify exactly how much damage a documented noise model does, and test whether a mitigation trick really helps.

> **Extra package:** this challenge needs `qiskit-aer==0.18.0` (the setup cell installs it if missing).

### Learning objectives
- Build a noise model with gate (depolarising) and readout errors.
- Run ideal and noisy experiments under identical, reproducible settings.
- Quantify the deviation with the total variation distance.
- Evaluate - not assume - a simple mitigation technique.

### Prerequisites
- Challenges 1-2 or equivalent
- Measurement counts
- Qiskit Aer (installed as an optional extra)

### What you will write
| Function | Status |
|---|---|
| `build_noise_model` | required |
| `run_counts` | required |
| `total_variation_distance` | required |
| `noise_report` | required |
| `calibration_matrix` | bonus (optional) |
| `mitigate_readout` | bonus (optional) |
| `evaluate_mitigation` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `build_noise_model` attaches depolarising errors with the documented parameters to sx, x and cx and a symmetric readout error to every qubit.
- `run_counts` configures ideal and noisy runs consistently (same transpilation, shots and seed) and is reproducible.
- `total_variation_distance` is a correct distance between count dictionaries.
- `noise_report` quantifies noise: TVD against the exact ideal distribution and probability mass outside the ideal support.
- Stronger noise gives a larger measured deviation (controlled seed, statistical tolerance).
- Invalid probabilities, shots and empty inputs raise ValueError; `run_counts` does not modify the circuit it is given.

**Bonus (optional, never required to pass):**
- `calibration_matrix` / `mitigate_readout` invert a known readout channel exactly and return valid distributions.
- `evaluate_mitigation` reports measured numbers: mitigation helps for readout-only noise and is not claimed to fix gate noise.

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

# Colab opens a notebook straight from GitHub without the project next to it: the project is fetched from this repository.
PROJECT_REPOSITORY = 'https://github.com/hastikacheddy/qiskit-fall-fest-participant.git'

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    try:
        on_colab = importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:  # there is no `google` package at all
        on_colab = False
    if root is None and on_colab:
        # the notebook is here, the project is not: fetch it (a few seconds)
        root = here / PROJECT_REPOSITORY.rsplit('/', 1)[-1].removesuffix('.git')
        if not root.exists():
            print('Fetching the project from', PROJECT_REPOSITORY, '...')
            try:
                subprocess.check_call(['git', 'clone', '--quiet', '--depth', '1', PROJECT_REPOSITORY, str(root)])
            except (subprocess.CalledProcessError, OSError) as error:
                raise RuntimeError(f'Could not fetch the project from {PROJECT_REPOSITORY} ({error}). '
                                   "Upload the project yourself instead (see README.md, section 'Colab').") from None
    if root is None or not (root / 'pyproject.toml').is_file():
        raise RuntimeError('Open this notebook from inside the project folder (the one that contains pyproject.toml). '
                           'If you only have the notebook, download the whole project from ' + PROJECT_REPOSITORY.removesuffix('.git') +
                           " (Code > Download ZIP) or open the notebook in Google Colab - see README.md, section 'Colab'.")
    print('Installing the project and its libraries - about a minute the first time ...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()
# This challenge needs an optional package:
from qiskit_fest.env import ensure_packages
ensure_packages('qiskit-aer==0.18.0')


## The problem
Your circuit works perfectly - on paper. A real quantum computer makes small errors with every gate and every measurement. In this challenge you **quantify exactly how much damage a documented noise model does** to a baseline circuit - reproducibly - and, as a bonus, you **test instead of assume** whether a mitigation trick helps.

1. `build_noise_model(p1, p2, p_readout)` - build the documented noise model with Qiskit Aer.
2. `run_counts(circuit, noise_model, shots, seed)` - run a circuit ideal (no noise model) or noisy, under *identical and reproducible* settings.
3. `total_variation_distance(counts_a, counts_b)` - a distance between two outcome histograms.
4. `noise_report(circuit, noise_model, shots, seed)` - quantify the noise: deviation from the exact ideal distribution and probability mass in outcomes that can only occur because of noise.
5. *(Bonus)* `calibration_matrix`, `mitigate_readout`, `evaluate_mitigation` - readout-error mitigation, judged by measured numbers.

### The documented noise model
| Where | Error |
|---|---|
| after every `sx` and every `x` gate, on any qubit | depolarising error with parameter **p1** |
| after every `cx` gate, on any ordered pair of qubits | two-qubit depolarising error with parameter **p2** |
| when a qubit is measured | symmetric readout error: P(read 1 &#124; qubit is 0) = P(read 0 &#124; qubit is 1) = **p_readout**, independently for every qubit |
| everything else, e.g. `rz` (a *virtual* gate on IBM hardware) | noiseless |

A *depolarising error with parameter p on k qubits* is the channel ρ → (1 − p)·ρ + p·(I/2ᵏ ⊗ Tr_S ρ) (S = the k qubits it acts on). It is exactly what `qiskit_aer.noise.depolarizing_error(p, k)` implements. All three parameters are real numbers in [0, 1]. The values `DOCUMENTED_NOISE = {p1: 0.002, p2: 0.015, p_readout: 0.02}` have the order of magnitude of today's superconducting devices; they are **not** a model of any specific chip.

### Run configuration (identical for ideal and noisy runs)
`run_counts` always transpiles with `generate_preset_pass_manager(optimization_level=0, basis_gates=BASIS_GATES, seed_transpiler=seed)`, where `BASIS_GATES = ['id', 'rz', 'sx', 'x', 'cx']`, and runs the result on `AerSimulator(noise_model=noise_model, seed_simulator=seed)` with the requested number of shots (`noise_model=None` means ideal). A noise model is attached to gate *names*, so the circuit must be rewritten into the gates the model talks about - in **both** runs, otherwise the comparison is not fair.

### Function contracts
| Function | Input | Output | Raises `ValueError` when |
|---|---|---|---|
| `build_noise_model(p1, p2, p_readout)` | three real numbers in [0, 1] | `qiskit_aer.noise.NoiseModel` exactly as documented above | a parameter is not a real number in [0, 1] (NaN included) |
| `run_counts(circuit, noise_model=None, shots=4000, seed=1234)` | circuit whose measurements are at the end (one classical register), a `NoiseModel` or `None`, a positive `int`, an `int` | `dict` bit-string → count (Qiskit bit order, counts add up to `shots`). Equal arguments give identical output. The `circuit` you pass in is **not modified** | `shots` is not a positive integer, or the circuit contains no measurement |
| `total_variation_distance(counts_a, counts_b)` | two dicts bit-string → non-negative number (counts *or* probabilities) | float in [0, 1]: ½·Σ&#124;p(x) − q(x)&#124; after dividing each dict by its own total; a missing key counts as 0 | a dict is empty, has a negative or non-finite value, or sums to 0 |
| `noise_report(circuit, noise_model, shots=4000, seed=1234)` | as for `run_counts` | `dict` with the keys listed below | as for `run_counts` |
| *(bonus)* `calibration_matrix(n, noise_model, shots=4000, seed=1234)` | integer 1 ≤ n ≤ 6 | NumPy array, shape (2ⁿ, 2ⁿ): column *j* = measured probabilities after preparing basis state *j* (X on qubit *q* iff bit *q* of *j* is 1) and measuring all qubits via `run_counts`; entry [i, j] = P(read i &#124; prepared j) with i, j = `int(bit-string, 2)` | *n* is not an integer in [1, 6] |
| *(bonus)* `mitigate_readout(counts, matrix)` | counts or probabilities of n-bit strings; a (2ⁿ, 2ⁿ) matrix | `dict` over **all** 2ⁿ bit-strings: solve `matrix · q = p_measured`, set negative entries to 0, divide by the sum | shapes do not match, matrix singular (condition number > 10¹²), counts empty or invalid |
| *(bonus)* `evaluate_mitigation(circuit, noise_model, shots=4000, seed=1234)` | as for `run_counts` | `dict` with `tvd_raw`, `tvd_mitigated` and `improvement = tvd_raw − tvd_mitigated`, all measured against the exact ideal distribution | as for `run_counts` |

**The keys of `noise_report`:** `counts_ideal` = `run_counts(circuit, None, shots, seed)`; `counts_noisy` = `run_counts(circuit, noise_model, shots, seed)`; `tvd_noisy` = TVD between `counts_noisy` and the **exact** ideal distribution (`ideal_distribution(circuit)`, provided); `tvd_ideal` = the same for `counts_ideal` - the **shot-noise floor**: even a perfect machine does not reproduce a distribution exactly with finitely many shots; `outside_support` = fraction of the noisy shots whose bit-string has ideal probability 0.

**Randomness and tolerances.** Counts are random, so the tests never demand one exact histogram. They check (i) deterministic quantities *exactly* (the distance between two given dictionaries, the exact channel of one gate, equal seeds giving equal counts) and (ii) sampled counts against the exact expected distribution with **exact binomial intervals** (a correct implementation violates one with probability below 10⁻⁷ per outcome).

## Concept: noise, shots and distance
* **Gate errors.** Every physical gate is slightly wrong. The *depolarising* model says: with probability p the qubit(s) are replaced by the maximally mixed state. After an `x` gate on |0⟩ the probability of finding |1⟩ is therefore 1 − p/2 (not 1).
* **Readout errors.** Measuring |0⟩ may report 1 and vice versa; the model is a 2×2 table of probabilities `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]` (rows = what the qubit *really* was).
* **Shot noise.** Even without any noise, N shots give frequencies that scatter by about √(p(1−p)/N) around the true probabilities. A deviation must be compared with this floor before you call it noise.
* **Total variation distance** $\mathrm{TVD}(p,q)=\tfrac12\sum_x|p(x)-q(x)|$: the largest probability difference any single event can show (0 = identical, 1 = disjoint).
* **Why transpile?** A noise model attaches errors to gate names. The Hadamard `h` is not a hardware gate; after transpilation it becomes `rz·sx·rz` and the `sx` carries the error. Without transpilation the same circuit would silently be *less noisy*.
* **Seeds.** Fixing the simulator seed makes a noisy run reproducible; fixing the *same* seed for ideal and noisy runs makes them comparable.

In [ ]:
import math
import numbers

import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error
from qiskit_fest.challenges.c06_noise import BASIS_GATES, ideal_distribution

In [ ]:
from qiskit_fest.challenges.c06_noise import (
    DOCUMENTED_NOISE, baseline_circuit, noise_scales, plot_histograms, plot_mitigation, plot_tvd_sweep, probe_readout, probe_x_gate, tvd_sweep)
from qiskit_fest.viz import draw_circuit

circuit = baseline_circuit(3)
draw_circuit(circuit, "Baseline circuit: a 3-qubit GHZ state, then measure everything")
exact = ideal_distribution(circuit)
print("exact ideal distribution (no shots, no noise):", {k: round(v, 3) for k, v in exact.items()})
print("basis gates the circuit is transpiled to:", BASIS_GATES)
print("documented noise parameters:", DOCUMENTED_NOISE)

## Part 1 · The noise model  *(TODO)*
Complete `build_noise_model(p1, p2, p_readout)` exactly as documented. Two probes check your model *without any circuit of yours*:
`probe_x_gate(model)` is the exact probability of |1⟩ after one `x` gate, `probe_readout(model)` is the fraction of wrong readings of a |0⟩ qubit.

<details><summary>💡 Hint 1 (subtle)</summary>

A `NoiseModel` is a collection of *errors*, each attached to gate **names**; the simulator applies an error whenever it executes a gate with that name. Which gate names does a circuit contain after it has been transpiled to `BASIS_GATES`?
</details>

<details><summary>💡 Hint 2</summary>

Building blocks: `depolarizing_error(param, num_qubits)`; `ReadoutError([[P(0|0), P(1|0)], [P(0|1), P(1|1)]])` (each row = one *true* value); the methods `noise_model.add_all_qubit_quantum_error(error, gate_names)` and `noise_model.add_all_qubit_readout_error(error)`. Start with `NoiseModel(basis_gates=BASIS_GATES)`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

```python
noise_model = NoiseModel(basis_gates=BASIS_GATES)
noise_model.add_all_qubit_quantum_error(depolarizing_error(___, 1), ["sx", "x"])
noise_model.add_all_qubit_quantum_error(depolarizing_error(___, 2), ["cx"])
noise_model.add_all_qubit_readout_error(ReadoutError([[1 - ___, ___], [___, 1 - ___]]))
```
Validate the three numbers first (`nan` fails every comparison, so `0 <= p <= 1` rejects it too).
</details>

In [ ]:
def build_noise_model(p1: float, p2: float, p_readout: float) -> NoiseModel:
    """Noise model of the documentation: depolarising p1 after sx/x, depolarising p2 after cx, symmetric readout error p_readout."""
    # TODO: validate the parameters (ValueError), then build and return the NoiseModel.
    raise NotImplementedError("TODO: implement build_noise_model()")

In [ ]:
with if_ready():
    model = build_noise_model(0.2, 0.0, 0.1)
    print(f"one x gate with p1 = 0.2: exact P(|1>) = {probe_x_gate(model):.4f}    (documentation: 1 - p1/2 = 0.9000)")
    print(f"measuring |0> with p_readout = 0.1 reads '1' in {probe_readout(model):.3f} of 20000 shots    (documentation: 0.100, shot noise about +-0.002)")

In [ ]:
report = run_public_checks(6, globals(), keyword="build_noise_model")

## Part 2 · Ideal and noisy runs under the same settings  *(TODO)*
Complete `run_counts(circuit, noise_model, shots, seed)`. Follow the **run configuration** of the problem statement - the *same* transpilation, shots and seed with and without noise - and return a plain `dict` of counts.

<details><summary>💡 Hint 1 (subtle)</summary>

The noise model is attached to gate *names*, and your circuit contains `h`. What has to happen to the circuit before the simulator runs it - in the ideal *and* in the noisy run? Also: which of your objects does a transpiler modify, and which does it copy?
</details>

<details><summary>💡 Hint 2</summary>

`generate_preset_pass_manager(optimization_level=0, basis_gates=BASIS_GATES, seed_transpiler=seed).run(circuit)` returns a transpiled **copy**. Create the backend with `AerSimulator(noise_model=noise_model, seed_simulator=seed)` (`noise_model=None` is fine) and give `run` the shots and the seed too. `result.get_counts()` returns a dict-like object.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

1. Validate: `shots` must be a positive integer; the circuit must contain at least one instruction named `"measure"` (`ValueError` otherwise).
2. `transpiled = pass_manager.run(circuit)`.
3. `result = AerSimulator(noise_model=___, seed_simulator=___).run(transpiled, shots=___, seed_simulator=___).result()`.
4. `return dict(result.get_counts())`.
</details>

In [ ]:
def run_counts(circuit: QuantumCircuit, noise_model: NoiseModel | None = None, shots: int = 4000, seed: int = 1234) -> dict[str, int]:
    """Counts {bit-string: count} of `circuit` on AerSimulator; noise_model=None means ideal. Same transpilation, shots and seed always."""
    # TODO: validate, transpile (basis gates, optimization level 0, seed), run with the seed, return the counts as a dict.
    raise NotImplementedError("TODO: implement run_counts()")

In [ ]:
with if_ready():
    ideal_counts = run_counts(circuit, None, shots=4000, seed=1234)
    noisy_counts = run_counts(circuit, build_noise_model(**DOCUMENTED_NOISE), shots=4000, seed=1234)
    print("ideal run:", dict(sorted(ideal_counts.items())))
    print("noisy run:", dict(sorted(noisy_counts.items())))
    plot_histograms(exact, ideal_counts, noisy_counts, "GHZ state: exact, ideal run and noisy run")

In [ ]:
report = run_public_checks(6, globals(), keyword="run_counts")

## Part 3 · A distance between histograms  *(TODO)*
Complete `total_variation_distance(counts_a, counts_b)`: both dictionaries are first divided by their own total (counts → probabilities), then TVD = ½·Σ|p(x) − q(x)| over all bit-strings that occur in either dictionary.

<details><summary>💡 Hint 1 (subtle)</summary>

Counts of 70/30 and 700/300 describe the same distribution. What has to happen before you compare two histograms that were recorded with different numbers of shots?
</details>

<details><summary>💡 Hint 2</summary>

Loop over `set(a) | set(b)`; `dict.get(key, 0.0)` gives 0 for a bit-string that one histogram never saw. Reject an empty dictionary, a negative or non-finite value and a total of 0 with `ValueError`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`p = {k: v / sum(a.values()) for k, v in a.items()}` (same for `b` → `q`), then `0.5 * sum(abs(p.get(k, 0.0) - q.get(k, 0.0)) for k in set(p) | set(q))`.
</details>

In [ ]:
def total_variation_distance(counts_a: dict, counts_b: dict) -> float:
    """TVD = 1/2 sum |p - q| of two histograms (each divided by its own total; missing keys count as 0)."""
    # TODO: validate (ValueError), normalise both dictionaries, return half the L1 distance.
    raise NotImplementedError("TODO: implement total_variation_distance()")

In [ ]:
with if_ready():
    print("toy example, 70/30 against 50/50:", round(total_variation_distance({"0": 70, "1": 30}, {"0": 50, "1": 50}), 4), "(by hand: 0.2)")
    print("counts are normalised, so 7/3 and 700/300 are the same histogram:", total_variation_distance({"0": 7, "1": 3}, {"0": 700, "1": 300}))
    print("disjoint supports:", total_variation_distance({"00": 5}, {"11": 9}), "(must be 1)")

In [ ]:
report = run_public_checks(6, globals(), keyword="total_variation_distance")

## Part 4 · A noise report  *(TODO)*
Complete `noise_report(circuit, noise_model, shots, seed)`. Run the circuit ideal and noisy with the same shots and seed (**your** `run_counts`), compare both with the **exact** ideal distribution `ideal_distribution(circuit)` and return the dictionary described in the problem statement.

<details><summary>💡 Hint 1 (subtle)</summary>

You need three kinds of numbers: the deviation of the noisy run from the exact ideal distribution, the same number for the *ideal* run (what does it tell you?), and the fraction of shots that landed on outcomes the ideal circuit can never produce.
</details>

<details><summary>💡 Hint 2</summary>

The *support* of the ideal distribution is the set of its keys. Call `run_counts` twice - `noise_model=None` for the ideal run - with the same `shots` and `seed`; compare each result with `exact` using your TVD.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`outside_support = sum(c for bits, c in counts_noisy.items() if bits not in set(exact)) / sum(counts_noisy.values())`; return a dict with the five keys `counts_ideal`, `counts_noisy`, `tvd_ideal`, `tvd_noisy`, `outside_support`.
</details>

In [ ]:
def noise_report(circuit: QuantumCircuit, noise_model: NoiseModel, shots: int = 4000, seed: int = 1234) -> dict:
    """Dict with counts_ideal, counts_noisy, tvd_ideal, tvd_noisy (both against the exact ideal distribution) and outside_support."""
    # TODO: exact ideal distribution, two runs with run_counts, two TVDs, the probability mass outside the ideal support.
    raise NotImplementedError("TODO: implement noise_report()")

In [ ]:
with if_ready():
    report = noise_report(circuit, build_noise_model(**DOCUMENTED_NOISE), shots=4000, seed=1234)
    print(f"shot-noise floor  tvd_ideal = {report['tvd_ideal']:.4f}   (a perfect machine, 4000 shots)")
    print(f"noisy run         tvd_noisy = {report['tvd_noisy']:.4f}   (documented noise)")
    print(f"mass outside the ideal support = {report['outside_support']:.4f}")

In [ ]:
report = run_public_checks(6, globals(), keyword="noise_report")

## Part 5 · Quantify and interpret  *(not graded)*
With your functions you can now run real experiments. The plot shows how the deviation grows with the noise strength; the dashed line is the shot-noise floor of the ideal runs. The table separates *where* the damage comes from.
Replace the baseline by other circuits (`baseline_circuit(n)` for n = 2 … 6, or your own with `measure_all()`).

In [ ]:
with if_ready():
    sweep = tvd_sweep(circuit, noise_report, build_noise_model)
    plot_tvd_sweep(sweep)
    print(f"{'noise source':24s} {'TVD to the exact distribution':>30s}")
    for label, parts in [("gate errors only", ("p1", "p2")), ("readout error only", ("p_readout",)), ("everything (documented)", ("p1", "p2", "p_readout"))]:
        rep = noise_report(circuit, build_noise_model(**noise_scales(None, 1.0, parts)), shots=4000, seed=1234)
        print(f"{label:24s} {rep['tvd_noisy']:>30.4f}")

## Bonus · Readout-error mitigation  *(optional)*
Readout errors are the same in every experiment, so they can be *measured* and *inverted*:
1. **Calibrate:** prepare each basis state |j⟩, measure, and record the confusion matrix `A[i][j] = P(read i | prepared j)`.
2. **Invert:** the measured distribution is `A · p_true`, so `p_true ≈ A⁻¹ · p_measured`. Statistical noise can make entries slightly negative - set them to 0 and renormalise.
3. **Evaluate - do not assume:** compare TVDs with and without mitigation for *different* kinds of noise. Mitigation can only repair what the calibration matrix describes.

<details><summary>💡 Hint</summary>

Prepare basis state `j` with `qc.x(q)` for every qubit `q` whose bit `(j >> q) & 1` is 1, then `qc.measure_all()`, then `run_counts`. Fill column `j` with `count / shots` at row `int(bit-string, 2)`. For the inverse use `np.linalg.solve(matrix, p_measured)`.
</details>

In [ ]:
def calibration_matrix(n: int, noise_model: NoiseModel | None, shots: int = 4000, seed: int = 1234) -> np.ndarray:
    """(2**n, 2**n) confusion matrix: column j = measured probabilities after preparing basis state j (Qiskit bit order)."""
    # TODO (bonus): 2**n calibration circuits, run each with run_counts, fill the columns.
    raise NotImplementedError("TODO (bonus): implement calibration_matrix()")

In [ ]:
def mitigate_readout(counts: dict, matrix: np.ndarray) -> dict[str, float]:
    """Probabilities over all 2**n bit-strings: solve matrix @ q = p_measured, set negative entries to 0, renormalise."""
    # TODO (bonus): validate shapes and conditioning, solve, clip, renormalise.
    raise NotImplementedError("TODO (bonus): implement mitigate_readout()")

In [ ]:
def evaluate_mitigation(circuit: QuantumCircuit, noise_model: NoiseModel | None, shots: int = 4000, seed: int = 1234) -> dict:
    """Measured effect of readout mitigation: tvd_raw, tvd_mitigated, improvement = tvd_raw - tvd_mitigated (vs the exact ideal distribution)."""
    # TODO (bonus): raw run, calibration, mitigation, two TVDs against the exact ideal distribution. Report numbers, do not assume success.
    raise NotImplementedError("TODO (bonus): implement evaluate_mitigation()")

In [ ]:
with if_ready():
    print(f"{'noise (documented x2)':26s} {'TVD raw':>9s} {'TVD mitigated':>14s} {'improvement':>12s}")
    for label, parts in [("readout error only", ("p_readout",)), ("gate errors only", ("p1", "p2")), ("everything", ("p1", "p2", "p_readout"))]:
        r = evaluate_mitigation(circuit, build_noise_model(**noise_scales(None, 2.0, parts)), shots=4000, seed=1234)
        print(f"{label:26s} {r['tvd_raw']:>9.4f} {r['tvd_mitigated']:>14.4f} {r['improvement']:>12.4f}")
    model = build_noise_model(**noise_scales(None, 2.0, ("p_readout",)))
    raw_counts = run_counts(circuit, model, 4000, 1234)
    plot_mitigation(exact, raw_counts, mitigate_readout(raw_counts, calibration_matrix(3, model, 4000, 1234)), "Readout-only noise: raw versus mitigated")

In [ ]:
report = run_public_checks(6, globals(), keyword="bonus")

## Reflection
1. Why do we compare the noisy counts with the **exact** ideal distribution and not with the counts of an ideal run? What does `tvd_ideal` tell you, and how should it influence the way you read `tvd_noisy`?
2. For the baseline GHZ circuit: which noise source dominates - gates or readout? How could you tell with the table of Part 5 alone? What would change for a circuit with many more CNOT gates?
3. Run the experiments with five different seeds. How large is the spread of `tvd_noisy`, and how many shots would you need to make it half as large?
4. (Bonus) Why can mitigation help although it cannot fix gate errors? What does it cost for *n* qubits, and what can go wrong with the inverse of the calibration matrix?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 6)